In [138]:
#For Connecting to MS SQL Server
! pip install pyodbc

#For converting portugese to english
! pip install unidecode

import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

## **Understanding Data**

## **1. olist_customers_dataset.csv:**
customer_id: unique identifier for each customer
customer_unique_id: unique identifier for each customer (anonymized)
customer_zip_code_prefix: zip code prefix of the customer's address
customer_city: city where the customer is located
customer_state: state where the customer is located
## **2. olist_geolocation_dataset.csv:**
geolocation_zip_code_prefix: zip code prefix for the location
geolocation_lat: latitude of the location
geolocation_lng: longitude of the location
geolocation_city: city of the location
geolocation_state: state of the location
## **3. olist_orders_dataset.csv:**
order_id: unique identifier for each order
customer_id: unique identifier for the customer who placed the order
order_status: current status of the order (e.g. delivered, shipped, canceled)
order_purchase_timestamp: date and time when the order was placed
order_approved_at: date and time when the payment for the order was approved
order_delivered_carrier_date: date and time when the order was handed over to the carrier
order_delivered_customer_date: date and time when the order was delivered to the customer
order_estimated_delivery_date: estimated date when the order is expected to be delivered
## **4. olist_order_items_dataset.csv:**
order_id: unique identifier for the order
order_item_id: unique identifier for each item within an order
product_id: unique identifier for the product being ordered
seller_id: unique identifier for the seller who listed the product
shipping_limit_date: date and time when the seller has to ship the product
price: price of the product
freight_value: shipping fee for the product
## **5. olist_order_payments_dataset.csv:**
order_id: unique identifier for the order
payment_sequential: index number for each payment made for an order
payment_type: type of payment used for the order (e.g. credit card, debit card, voucher)
payment_installments: number of installments in which the payment was made
payment_value: value of the payment made
## **6. olist_products_dataset.csv:**
product_id: unique identifier for each product
product_category_name: name of the category that the product belongs to
product_name_lenght: number of characters in the product name
product_description_lenght: number of characters in the product description
product_photos_qty: number of photos for the product
product_weight_g: weight of the product in grams
product_length_cm: length of the product in centimeters
product_height_cm: height of the product in centimeters
product_width_cm: width of the product in centimeters
## **7. olist_sellers_dataset.csv:**
seller_id: unique identifier for each seller
seller_zip_code_prefix: zip code prefix for the seller's location
seller_city: city where the seller is located
seller_state: state where the seller is located
## **8. product_category_name_translation.csv:**
product_category_name: name of the product category in Portuguese
product_category_name_english: name of the product category in English
## **9. olist_order_reviews_dataset.csv:**
review_id: unique identifier for each review
order_id: unique identifier for the order that the review is associated with
review_score: numerical score (1-5) given by the customer for the product
review_comment_title: title of the review comment
review_comment_message: text of the review comment
review_creation_date: date and time when the review was created
review_answer_timestamp: date and time when the seller responded to the review (if
applicable)
Note: The review comment fields (i.e. review_comment_title and review_comment_message)
are optional, and may not be present in all reviews.

## **Data Cleaning**

In [139]:
server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

In [140]:
query="select * from olist_customers_dataset"
customers= pd.read_sql(query, conn)

query="select * from olist_geolocation_dataset"
geolocation= pd.read_sql(query, conn)

query ="select * from olist_order_items_dataset"
olist_items= pd.read_sql(query, conn)

query ="select * from olist_order_payments_dataset"
payments= pd.read_sql(query, conn)

query ="select * from olist_order_reviews_dataset"
reviews= pd.read_sql(query, conn)

query ="select * from olist_orders_dataset"
orders= pd.read_sql(query, conn)

query ="select * from olist_products_dataset"
products= pd.read_sql(query, conn)

query ="select * from olist_sellers_dataset"
sellers= pd.read_sql(query, conn)

query ="select * from product_category_name_translation"
product_category_name_translation= pd.read_sql(query, conn)

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\2930530251.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers= pd.read_sql(query, conn)
C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\2930530251.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  geolocation= pd.read_sql(query, conn)
C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\2930530251.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  olist_items= pd.read_sql(query, conn)
C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\2930530251.py:11: UserWarnin

In [141]:
#Cleaning Functions
def converting_to_english(temp_df,column):
    "Converts to english language"
    temp_df[column] = temp_df[column].apply(unidecode)

    return temp_df


def keep_text(temp_df,column_list):
    "Removes unwanted spaces and special characters from textual columns"

    for col in column_list:
        temp_df[col] = temp_df[col].str.strip()
        temp_df[col] = temp_df[col].str.replace(r'[\s+]',' ',regex=True)
        temp_df[col] = temp_df[col].str.replace(r'[^A-Za-z\s]','',regex=True)

    return temp_df 

## **Customers**

**Customer Table Cleaning**

1. Converted customer_city Column to Title Case
Standardized the city names by converting all values in the customer_city column to Title Case (first letter of each word capitalized).
This ensured consistency in city name representation across the dataset.
Example:
sao paulo → Sao Paulo
rio de janeiro → Rio De Janeiro
Improved readability and reduced duplicate records caused by inconsistent text formatting.
2. Translated customer_city Values from Portuguese to English
Converted city names that were stored in Portuguese into their English equivalents wherever applicable.
This standardization made the dataset easier to understand and analyze for English-speaking users and stakeholders.
Helped maintain a common language format across the dataset, improving reporting and visualization consistency.
Example:
Portuguese city names or accented variations were normalized into standardized English-readable formats.
3. Removed Extra Spaces and Special Characters from customer_city
Cleaned the customer_city column by removing:
Leading and trailing spaces
Multiple consecutive spaces
Special characters and unwanted symbols
Accented characters where necessary for standardization
This reduced data quality issues caused by inconsistent user inputs and encoding differences.
Example:
" São-Paulo " → "Sao Paulo"
"Rio@de#Janeiro" → "Rio de Janeiro"
Improved matching accuracy during joins, aggregations, and location-based analysis.
4. Dropped the customer_id Column
Removed the customer_id column from the cleaned dataset as it was not required for the intended analysis or reporting purposes.
Eliminated unnecessary data, reducing dataset size and complexity.

customers.sample(10)

In [142]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 11.0 MB


In [143]:
customers.isna().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

In [144]:
customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')

In [145]:
print(customers['customer_city'].nunique())

print(customers['customer_state'].nunique())

4119
27


In [146]:
customers['customer_state'].unique()

<ArrowStringArray>
['SP', 'MG', 'ES', 'RJ', 'RS', 'BA', 'CE', 'PR', 'MS', 'PB', 'SC', 'MT', 'PA',
 'RN', 'PI', 'DF', 'GO', 'PE', 'RO', 'MA', 'SE', 'AM', 'AL', 'TO', 'AC', 'AP',
 'RR']
Length: 27, dtype: str

In [147]:
customers.duplicated().sum().item()

0

In [148]:
#Converting to title case
customers['customer_city'] = customers['customer_city'].str.title()

#Converting Portugese to English
customers = converting_to_english(customers,'customer_city')

#Removing special characters
customers = keep_text(customers,['customer_city'])

#Droping column
customers.drop(columns=['customer_unique_id'],inplace=True)

#Remving spaces
customers['customer_id'] = customers['customer_id'].str.strip()


In [149]:
customers.duplicated().sum().item()

0

In [150]:
customers.drop_duplicates(inplace=True)

In [151]:
customers.shape

(99441, 4)

In [152]:
customers.sample(10)

,customer_id,customer_zip_code_prefix,customer_city,customer_state
34849,5a1523d32a6d1b323a05895f5164655d,11065,Santos,SP
56983,92e944661c1d4cbed22d076c56a0c811,72460,Brasilia,DF
95987,f6ec7c6e0151f752c79e2f026a90da12,3274,Sao Paulo,SP
89699,e673336c4f14e065f1f05d154ae50993,4671,Sao Paulo,SP
81299,d0a06220865cc230d39188471bffd4f6,9920,Diadema,SP
71490,b77ee5671feb9b6efce84cc804842024,73088,Brasilia,DF
24201,3e46d957b84b38b6d616d4a486ef89a2,30692,Belo Horizonte,MG
5614,0e97ad6542c12412f0324dd07cb0daed,1503,Sao Paulo,SP
47408,7a575572300875aaf8a93e3c5772d76c,4209,Sao Paulo,SP
55903,902dbd012843fe49743fce302a7765fe,73050,Brasilia,DF


## **Geolocation**

**Geolocation Table Cleaning**
1. Dropped geolocation_lat and geolocation_lng Columns
Removed the geolocation_lat (latitude) and geolocation_lng (longitude) columns from the geolocation table.
These columns were not required for the scope of the analysis and were therefore excluded to simplify the dataset.
Eliminated unnecessary data attributes, reducing storage and processing overhead.
Improved dataset manageability by retaining only the fields relevant to business analysis and reporting.
2. Removed Special Characters and Extra Spaces from geolocation_city
Cleaned the geolocation_city column by removing unwanted special characters, symbols, and extra spaces.
Standardized city names to ensure consistency across all records.
Corrected formatting issues caused by data entry errors, encoding inconsistencies, or imported data variations.
Example:
" São-Paulo " → "Sao Paulo"
"Rio@de#Janeiro" → "Rio de Janeiro"
Enhanced data quality and improved the accuracy of grouping, filtering, and matching operations.
3. Converted geolocation_city Values to Title Case
Standardized city names by converting all values in the geolocation_city column to Title Case.
Ensured that the first letter of each word was capitalized while the remaining letters were converted to lowercase.
Improved readability and maintained a uniform naming convention throughout the dataset.
Example:
sao paulo → Sao Paulo
rio de janeiro → Rio De Janeiro
Reduced inconsistencies that could lead to duplicate city records during analysis.
4. Translated geolocation_city Values from Portuguese to English
Converted city names and location values from Portuguese to their English-standardized representations where applicable.
Improved accessibility and understanding of the dataset for English-speaking users and stakeholders.
Established a consistent language format across the geolocation data.
Facilitated easier interpretation, reporting, and visualization of geographic information.
Helped align geolocation data with other datasets that were maintained in English.

In [153]:
geolocation.sample(10)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
762922,20031,-22.908983,-43.177143,rio de janeiro,RJ
857799,26582,-22.791805,-43.414883,mesquita,RJ
843769,25585,-22.791115,-43.334053,são joão de meriti,RJ
231087,85400,-25.100416,-52.864414,guaraniacu,PR
962349,35931,-19.844894,-43.179127,joao monlevade,MG
899340,29960,-18.569912,-39.749981,conceicao da barra,ES
877205,28770,-21.959661,-42.012630,santa maria madalena,RJ
260301,88040,-27.608881,-48.526104,florianopolis,SC
896592,29680,-19.758726,-40.385693,joao neiva,ES
891907,29100,-20.329966,-40.292645,vila velha,ES


In [154]:
geolocation.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  str    
 1   geolocation_lat              998827 non-null   float64
 2   geolocation_lng              1000160 non-null  float64
 3   geolocation_city             1000163 non-null  str    
 4   geolocation_state            1000163 non-null  str    
dtypes: float64(2), str(3)
memory usage: 54.9 MB


In [155]:
geolocation.isna().sum()

geolocation_zip_code_prefix       0
geolocation_lat                1336
geolocation_lng                   3
geolocation_city                  0
geolocation_state                 0
dtype: int64

In [156]:
geolocation['geolocation_city'].nunique()

8011

In [157]:
geolocation[geolocation['geolocation_city'].str.contains(r'\d',regex=True)]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
144075,71880,-15.902908,-48.049046,riacho fundo 2,DF
181282,78278,-15.316965,-58.006672,lambari d%26apos%3boeste,MT
254629,87365,-24.271860,-53.069435,4o. centenario,PR
254873,87365,-24.281906,-53.074516,4º centenario,PR
255113,87365,-24.277697,-53.074932,4º centenario,PR
335671,96130,-31.699198,-52.155811,colônia z-3,RS
335683,96130,-31.697290,-52.156406,colônia z-3,RS
335709,96130,-31.698061,-52.156082,colonia z-3,RS
735269,17970,-21.269165,-51.668758,são joão do pau d%26apos%3balho,SP
896964,29735,-19.396948,-40.905041,quilometro 14 do mutum,ES


In [158]:
#Droping columns
geolocation.drop(columns=['geolocation_lat','geolocation_lng'],inplace=True)

#Converting to title case
geolocation['geolocation_city'] = geolocation['geolocation_city'].str.title()

#Converting from Portugese to English
geolocation = converting_to_english(geolocation,'geolocation_city')

#Removing special characters
geolocation = keep_text(geolocation,['geolocation_city'])

#Special case removel
geolocation["geolocation_city"]= geolocation["geolocation_city"].str.replace("Sao Joao Do Pau D Alho","Sao Joao Do Pau DAlho")


In [159]:
geolocation[geolocation['geolocation_city'].str.contains(r'\d',regex=True)]

,geolocation_zip_code_prefix,geolocation_city,geolocation_state


In [160]:
geolocation.duplicated().sum().item()

980551

In [161]:
geolocation.drop_duplicates(inplace=True)

In [162]:
geolocation.sample(10)

,geolocation_zip_code_prefix,geolocation_city,geolocation_state
455615,04610,Sao Paulo,SP
341145,96875,Gramado Xavier,RS
904403,30220,Belo Horizonte,MG
746416,18303,Capao Bonito,SP
900253,30180,Belo Horizonte,MG
300856,90510,Porto Alegre,RS
614886,11674,Caraguatatuba,SP
358107,01006,Sao Paulo,SP
678061,13801,MogiMirim,SP
39328,42835,Arembepe,BA


In [163]:
geolocation.shape

(19612, 3)

## **Product Category Translation**

**Product Table and Product Category Translation Table Cleaning – Data Preprocessing Step**
1. Dropped Unnecessary Product Attribute Columns
Removed the following columns from the product table:
product_name_lenght
product_description_lenght
product_weight_g
product_length_cm
product_height_cm
product_width_cm
These columns were not required for the intended analysis and were therefore excluded to simplify the dataset.
Reduced dataset complexity and storage requirements by eliminating attributes that did not contribute significant business value to the analysis.
Improved data processing efficiency and enabled focus on the most relevant product-related information.
2. Corrected Spelling Errors and Removed Spaces and Special Characters from product_category
Cleaned the product_category column by addressing data quality issues such as:
Spelling inconsistencies and typographical errors
Leading and trailing spaces
Multiple consecutive spaces
Special characters and unwanted symbols
Standardized category names to ensure uniform representation across the dataset.
Improved data consistency and prevented the creation of duplicate categories caused by formatting variations.
Enhanced the accuracy of category-based aggregations, filtering, and reporting.
Example:
" beleza_saude " → "beleza_saude"
Categories with inconsistent formatting were normalized into a standardized structure.
3. Merged Product Table with Product Category English Translation Table
Joined the product table with the product category English translation table using the product category field as the common key.
Retrieved the English translations corresponding to each Portuguese product category.
Replaced Portuguese category values with their English equivalents to create a more user-friendly and globally understandable dataset.
After successfully obtaining the English category names, the original Portuguese product category column was removed to avoid redundancy and maintain a single standardized category field.
This process improved data readability and ensured consistency across reports, dashboards, and analytical outputs.
Example:
beleza_saude → health_beauty
informatica_acessorios → computers_accessories

In [164]:
product_category_name_translation

,product_category_name,product_category_name_english
0,product_category_name,product_category_name_english
1,beleza_saude,health_beauty
2,informatica_acessorios,computers_accessories
3,automotivo,auto
4,cama_mesa_banho,bed_bath_table
...,...,...
67,flores,flowers
68,artes_e_artesanato,arts_and_craftmanship
69,fraldas_higiene,diapers_and_hygiene
70,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


In [165]:
product_category_name_translation.shape

(72, 2)

In [166]:
product_category_name_translation.duplicated().sum().item()

0

In [167]:
product_category_name_translation.isna().sum()

product_category_name            0
product_category_name_english    0
dtype: int64

In [168]:
product_category_name_translation.drop(index=0,inplace=True)

In [169]:
product_category_name_translation['product_category_name'].unique()

<ArrowStringArray>
[                                  'beleza_saude',
                         'informatica_acessorios',
                                     'automotivo',
                                'cama_mesa_banho',
                               'moveis_decoracao',
                                  'esporte_lazer',
                                     'perfumaria',
                          'utilidades_domesticas',
                                      'telefonia',
                             'relogios_presentes',
                              'alimentos_bebidas',
                                          'bebes',
                                      'papelaria',
                       'tablets_impressao_imagem',
                                     'brinquedos',
                                 'telefonia_fixa',
                             'ferramentas_jardim',
                    'fashion_bolsas_e_acessorios',
                                'eletroportateis',
            

In [170]:
product_category_name_translation['product_category_name_english'].unique()

<ArrowStringArray>
[                          'health_beauty',
                   'computers_accessories',
                                    'auto',
                          'bed_bath_table',
                         'furniture_decor',
                          'sports_leisure',
                               'perfumery',
                              'housewares',
                               'telephony',
                           'watches_gifts',
                              'food_drink',
                                    'baby',
                              'stationery',
                  'tablets_printing_image',
                                    'toys',
                         'fixed_telephony',
                            'garden_tools',
                'fashion_bags_accessories',
                        'small_appliances',
                          'consoles_games',
                                   'audio',
                           'fashion_shoes',
             

## **Products**

In [171]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700,31.0,13.0,20.0


In [172]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [173]:
#Droping columns
products.drop(columns=['product_name_lenght','product_description_lenght', 'product_weight_g',
       'product_length_cm', 'product_height_cm','product_width_cm'],inplace=True)

In [174]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   product_id             32951 non-null  str    
 1   product_category_name  32341 non-null  str    
 2   product_photos_qty     32341 non-null  float64
dtypes: float64(1), str(2)
memory usage: 2.2 MB


In [175]:
products

,product_id,product_category_name,product_photos_qty
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,1.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,1.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,1.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,1.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,4.0
...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,2.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,1.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,1.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,2.0


In [176]:
products.duplicated().sum().item()

0

In [177]:
products.isna().sum()

product_id                 0
product_category_name    610
product_photos_qty       610
dtype: int64

In [178]:
products[products['product_category_name'].isna()]

,product_id,product_category_name,product_photos_qty
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN
154,46b48281eb6d663ced748f324108c733,NaN,NaN
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN
...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN


In [179]:
products['product_category_name'].nunique()

73

In [180]:
product_category_name_translation['product_category_name'].nunique()

71

In [181]:
set_1 = set(products['product_category_name'].unique())


set_2 = set(product_category_name_translation['product_category_name'].unique())

set_1 - set_2

{nan, 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos'}

In [182]:
products = products.merge(product_category_name_translation,on='product_category_name',how='left')

In [183]:
products[products['product_category_name_english'].isna()]['product_category_name'].unique()

<ArrowStringArray>
[nan, 'pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']
Length: 3, dtype: str

In [184]:
#Replacing 'pc_gamer' with 'Gaming PC'
products.loc[products['product_category_name']=='pc_gamer','product_category_name_english']='Gaming PC'


#Replacing 'portateis_cozinha_e_preparadores_de_alimentos' with 'Portable Kitchen Appliances'
products.loc[products['product_category_name']=='portateis_cozinha_e_preparadores_de_alimentos','product_category_name_english']='Portable Kitchen Appliances'

In [185]:
products = products.drop(columns='product_category_name').rename(columns={'product_category_name_english':'product_category_name'}).reindex(columns=['product_id','product_category_name','product_photos_qty'])

In [186]:
products['product_category_name'] = products['product_category_name'].str.title().str.replace('_',' ').fillna('Other')

products['product_photos_qty'] = products['product_photos_qty'].fillna(1)

products['product_id'] = products['product_id'].str.strip()

In [187]:
products.isna().sum()

product_id               0
product_category_name    0
product_photos_qty       0
dtype: int64

In [188]:
products.sample(10)

,product_id,product_category_name,product_photos_qty
8754,cf30110b1e85017c00752838fec35442,Construction Tools Safety,1.0
13305,3dee4915c4189291d7c8b79da897ca3a,Furniture Decor,1.0
8136,ee33a2ccba10e33e8184c1b7be9ecf7d,Fixed Telephony,2.0
23152,39b86f4e3aedf22731990099a660a6f5,Fashion Bags Accessories,7.0
12853,b746f0f77468910428130facb2ee2d1f,Computers Accessories,3.0
293,f777c903946dd86fd00cd5b32ec3e907,Construction Tools Construction,4.0
6946,d14495a85be157b5cacef4eaaf825791,Kitchen Dining Laundry Garden Furniture,1.0
15596,5411e9269501a870cabf632f05655131,Stationery,3.0
26843,5bf0a3751ac1a1c1253ef97a027b8c13,Food,1.0
29180,02b7929be620a2f05bb13d0c0e2cbd48,Computers Accessories,3.0


In [189]:
products.shape

(32951, 3)

## **Sellers**

**Sellers Table Cleaning – Data Preprocessing Steps**   
Removed Spelling Mistakes, Unwanted Spaces, and Special Characters from seller_city
Cleaned the seller_city column by identifying and correcting spelling inconsistencies and typographical errors in city names.
Removed leading and trailing spaces, as well as multiple consecutive spaces between words, to ensure a uniform format.
Eliminated special characters, symbols, and unwanted punctuation that could affect data consistency and analysis.
Standardized city names to prevent duplicate entries caused by variations in spelling or formatting.
Example:
" Sao Paulo " → "Sao Paulo"
"Rio@de#Janeiro" → "Rio de Janeiro"
Improved the accuracy of city-based grouping, filtering, and reporting operations.
Enhanced data quality and ensured consistency when integrating seller data with other datasets containing geographic information.

In [190]:
sellers.sample(10)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
50,e546117ed9cafbc40239c0c78635c584,4705,sao paulo,SP
2754,3b872fd4747f01cc56206f2934198618,14940,ibitinga,SP
621,376d67b61dce0c990155286e7ae486a4,4815,sao paulo,SP
1922,cb32766839f443db85c82a43c4a6c19e,13506,rio claro,SP
356,fc0b214b59a83615fea981c6424a02ae,9380,maua,SP
2626,0249d282d911d23cb8b869ab49c99f53,5676,sao paulo,SP
340,6b89abe95848c850399130d149a39b63,75640,piracanjuba,GO
1560,062ce95fa2ad4dfaedfc79260130565f,95913,lajeado,RS
2274,a5a1bfcf728ab0e19182959cf0771ee4,13960,socorro,SP
970,5bc55dbe2f12b6af6d83ed46023e0dc8,35170,coronel fabriciano,MG


In [191]:
sellers.shape

(3095, 4)

In [192]:
sellers.info()

<class 'pandas.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   seller_id               3095 non-null   str  
 1   seller_zip_code_prefix  3095 non-null   int64
 2   seller_city             3095 non-null   str  
 3   seller_state            3095 non-null   str  
dtypes: int64(1), str(3)
memory usage: 230.3 KB


In [193]:
sellers.isna().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [194]:
sellers['seller_id'].duplicated().sum().item()

0

In [195]:
sellers['seller_id'].duplicated().sum().item()

0

In [196]:
sellers['seller_city'].nunique()

611

In [197]:
def convert_to_excel(temp_df,col_list):
    "Creates a copy of unique values in categorical columns into excel sheets"
    
    with pd.ExcelWriter("unique_values.xlsx") as writer:
        for col in col_list:
            pd.DataFrame(temp_df[col].unique(), columns=[col]).to_excel(
                writer,
                sheet_name=col,
                index=False
            )    

In [198]:
convert_to_excel(sellers,['seller_city'])

In [199]:
sellers[sellers['seller_city']=='04482255']

,seller_id,seller_zip_code_prefix,seller_city,seller_state
517,ceb7b4fb9401cd378de7886317ad1b47,22790,04482255,RJ


In [200]:
sellers['seller_id'] = sellers['seller_id'].str.strip()

sellers['seller_city'] = (
sellers['seller_city'].str.replace(r'[/,\\-]\s*\w*','',regex=True)
.str.strip()
.str.replace('04482255','Not Known')
.str.replace('vendas@creditparts.com.br','Not known')
.str.title()    
.str.replace(r'Sao Paulo  Paulo|Sao Pauo|Sao Paulo Sp|Sao Paulop|Sao Paluo','Sao Paulo',regex=True)
)

In [201]:
convert_to_excel(sellers,['seller_city'])

In [202]:
sellers['seller_state'].unique()

<ArrowStringArray>
['SP', 'RJ', 'PE', 'PR', 'GO', 'SC', 'BA', 'DF', 'RS', 'MG', 'RN', 'MT', 'CE',
 'PB', 'AC', 'ES', 'RO', 'PI', 'MS', 'SE', 'MA', 'AM', 'PA']
Length: 23, dtype: str

In [203]:
sellers.sample(10)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
1211,5305693ffae2d3463377b1f6fe67b15a,16300,Penapolis,SP
317,5acd070dd3fe441bbb2ec1f1ede515ee,13203,Jundiai,SP
613,f46490624488d3ff7ce78613913a7711,7194,Guarulhos,SP
127,99002261c568a84cce14d43fcffb43ea,78095,Cuiaba,MT
223,3d621842b2ed28e2b474132480edac3c,8041,Sao Paulo,SP
176,a63bfbaa882c8f4542891b4e2246cc7f,24355,Niteroi,RJ
1137,bdb3edbaee43a761e2d4f258dc08f348,18680,Lencois Paulista,SP
2799,b76a1f8356322f12529c37b67d5c96c2,11015,Santos,SP
2518,e58a5b390e28abc0b216cfb0e07d27d7,22640,Rio De Janeiro,RJ
430,c8c1bea22194a4eefa2dc9a9fa89f536,88210,Porto Belo,SC


In [204]:
sellers.shape

(3095, 4)

## **Orders**

**Orders Table Cleaning – Data Preprocessing Steps**
1. Removed Orders with unavailable Status and Consolidated Similar Order Statuses
Identified and removed all records where the order_status was marked as unavailable.
These orders represented transactions where customers placed orders for products that were already known to be unavailable, making them unsuitable for meaningful order lifecycle and fulfillment analysis.
Excluding such records helped ensure that the dataset reflected only valid and actionable customer orders.
Additionally, the order statuses invoiced and approved were combined and replaced with a single status, preprocessing.
This transformation reduced the number of distinct categories (cardinality) within the order_status column.
Grouping similar stages of the order lifecycle simplified analysis and reporting while preserving the overall business meaning of the order process.
Improved consistency and made status-based visualizations and aggregations easier to interpret.
2. Removed Invalid Order Records Based on Date Sequence Validation
Performed a comprehensive validation of order-related date fields to ensure that the order lifecycle followed a logical chronological sequence.
Removed records containing inconsistent or invalid date relationships that violated the expected business process flow.
The following validation checks were applied:
Order approval date earlier than order purchase date
Orders cannot be approved before they are placed.
Order approval date later than the date the order was handed over to the courier
An order must be approved before it can be dispatched for delivery.
Courier delivery date later than the customer delivery date
A package cannot be delivered to the customer before it has been handed over and processed through the delivery channel.
Records failing any of these conditions were identified as data quality issues and removed from the dataset.
This ensured that all retained orders followed a valid and realistic order processing timeline.
Improved the reliability of delivery performance analysis, fulfillment tracking, lead time calculations, and operational reporting.

In [205]:
orders.sample(10)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
4589,e1b286432c65d0a0e8032aefcdae26ba,e983a1272205857c12416007c4e6c9a2,delivered,2018-02-11 19:47:15,2018-02-11 20:08:02,2018-02-17 01:09:17,2018-03-06 23:04:50,2018-03-12
44659,632984d58b2a50f588c4a3c559c1f8f3,422fc5a15f629ad7a487384d91beec05,delivered,2017-06-28 18:13:18,2017-06-28 18:25:09,2017-06-30 15:36:04,2017-07-12 18:57:37,2017-07-24
30936,cb8add3d9d1e64220e64944c428eadc9,c83ceef725175bc8789fd9b820098b08,delivered,2017-12-02 11:04:57,2017-12-02 11:30:35,2017-12-06 10:12:12,2017-12-18 19:48:58,2018-01-02
552,df8e5e994bcc820fcf403f9a875201e6,05f1c2a8e913a4084727ac728bbdbec2,delivered,2018-08-01 13:30:12,2018-08-01 13:44:25,2018-08-02 14:20:00,2018-08-06 22:28:21,2018-08-17
50102,cbcffd62e7d1e6d26d3a6657731cb96a,343d05fdc8c888fa5ac9fdcea627efc6,delivered,2018-02-26 09:38:37,2018-02-26 09:55:50,2018-02-26 18:54:13,2018-04-16 23:55:37,2018-03-21
80309,72af7514238e36c16453f0a9146ecb52,1ea8e45ffebf367b2f046c8b3c8ff230,delivered,2017-12-01 15:40:46,2017-12-03 15:35:50,2017-12-04 22:52:24,2017-12-11 16:58:43,2017-12-19
77422,5d93a32a25ed8f8e884e5a1d283c43dd,b0dfff3b69c6420c45bee6e689f8b060,delivered,2018-06-19 11:23:30,2018-06-19 11:35:24,2018-06-20 14:43:00,2018-07-10 15:06:43,2018-07-17
96436,c77296df05b5ed399d2d111ec97b0b17,9ebea242c3341fd8563bb832e64a569e,delivered,2018-07-21 15:05:23,2018-07-21 15:15:17,2018-07-26 13:58:00,2018-08-09 16:45:41,2018-08-13
62352,619938bf55625c762a49ebba395e2eb5,55a1df4cbac7aa59f3bcdb5264128349,delivered,2017-11-29 01:40:17,2017-11-29 01:57:26,2017-11-30 14:54:00,2017-12-11 22:32:36,2017-12-20
12990,1aae876e8a8dcd3f41f6ec9aae3de9e3,adf8954ffb6abeb41d3ca91f01451710,delivered,2018-01-09 13:04:37,2018-01-10 10:35:34,2018-01-11 22:34:16,2018-01-22 19:22:33,2018-01-30


In [206]:
orders.shape

(99441, 8)

In [207]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 13.0 MB


In [208]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [209]:
orders.duplicated().sum()

np.int64(0)

In [210]:
orders['order_id'].duplicated().sum()

np.int64(0)

In [211]:
orders['customer_id'].duplicated().sum()

np.int64(0)

In [212]:
orders['order_status'].unique()

<ArrowStringArray>
[  'delivered',    'invoiced',     'shipped',  'processing', 'unavailable',
    'canceled',     'created',    'approved']
Length: 8, dtype: str

In [213]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [214]:
orders.query("order_status == 'processing'")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
128,15bed8e2fec7fdbadb186b57c46c92f2,f3f0e613e0bdb9c7cee75504f0f90679,processing,2017-09-03 14:22:03,2017-09-03 14:30:09,NaT,NaT,2017-10-03
324,d3c8851a6651eeff2f73b0e011ac45d0,957f8e082185574de25992dc659ebbc0,processing,2016-10-05 22:44:13,2016-10-06 15:51:05,NaT,NaT,2016-12-09
741,6a6c7d523fd59eb5bbefc007331af717,d954782ec6c0e911292c8a80757ef28d,processing,2017-11-24 20:09:33,2017-11-24 23:15:15,NaT,NaT,2017-12-20
1192,745e1d4a7f8c4b548881788d4113bb1d,7198d7088442e4ddfe553353d8ddc957,processing,2017-11-17 06:34:36,2017-11-18 02:15:40,NaT,NaT,2017-12-12
1516,1d52ba7197c7acebbb4f826f6585536f,c9c7fe860d602373a9e93f8bfe9d877a,processing,2017-02-13 18:32:55,2017-02-13 18:43:55,NaT,NaT,2017-04-04
...,...,...,...,...,...,...,...,...
97400,dcdfc540e42725663242bb884c28f0a6,38972104038aa68fcc61277dbf6e7ca9,processing,2017-10-30 10:46:44,2017-10-30 11:09:55,NaT,NaT,2017-11-23
97666,e471815e7114cdb474064f7dbb1a8b67,092c9316ae71b2fe43e526043f351967,processing,2017-12-20 11:00:02,2017-12-20 11:10:43,NaT,NaT,2018-02-02
98089,10951d02d64917a34959abeb8130601e,3e6754e591ff3568ccc5bf69a649918c,processing,2018-02-13 21:02:02,2018-02-15 04:11:21,NaT,NaT,2018-03-13
99140,aea0db338150b526dde24f6fd953a5ed,379a02efdc6a56bd27f99b95fc2f6c06,processing,2017-12-26 21:56:13,2017-12-26 22:05:26,NaT,NaT,2018-01-26


In [215]:
orders = orders[orders['order_status']!='unavailable']

In [216]:
orders['order_status'] = orders['order_status'].str.replace(r'invoiced|approved','processing',regex=True)

In [217]:
orders['order_status'].value_counts()

order_status
delivered     96478
shipped        1107
canceled        625
processing      617
created           5
Name: count, dtype: int64

In [218]:
def remove_invalid_order_id_rows(row):
    delivery_date = row['order_delivered_customer_date']
    check1 = row['order_purchase_timestamp']<=row['order_approved_at']
    check2 = row['order_approved_at']<=row['order_delivered_carrier_date']
    check3 = row['order_delivered_carrier_date']<=row['order_delivered_customer_date']

    if row['order_status']=='delivered':
        if check1 and check2 and check3:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='shipped':
        if check1 and check2:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='processing':
        if check1:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='canceled':
        if pd.isna(delivery_date):
            return 'Yes'
        else:
            return 'No'
    else:
        return 'Yes' 

In [219]:
orders['Valid'] = orders.apply(remove_invalid_order_id_rows,axis=1)


orders = orders.loc[orders['Valid'] == 'Yes']

orders.drop(columns = ['Valid','order_estimated_delivery_date'],inplace=True)

In [220]:
orders['order_id'].duplicated().sum()

np.int64(0)

In [221]:
orders['customer_id'].duplicated().sum()

np.int64(0)

In [222]:
orders['order_status'].unique()

<ArrowStringArray>
['delivered', 'processing', 'shipped', 'canceled', 'created']
Length: 5, dtype: str

In [223]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 146
order_delivered_carrier_date     1172
order_delivered_customer_date    2339
dtype: int64

In [224]:
orders['order_status'].value_counts()

order_status
delivered     95082
shipped        1098
canceled        619
processing      617
created           5
Name: count, dtype: int64

In [225]:
orders[
(orders['order_purchase_timestamp']>orders['order_approved_at'])&
(orders['order_approved_at']>orders['order_delivered_carrier_date'])&
(orders['order_delivered_carrier_date']>orders['order_delivered_customer_date'])&
(orders['order_status']=='delivered')
    ]    

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date


In [226]:
orders['order_id'] = orders['order_id'].str.strip()

In [227]:
orders.shape

(97421, 7)

## **Payments**

**Payments Table Cleaning – Data Preprocessing Steps**
1. Reduced Row Granularity by Aggregating Payment Information
The payments table contained multiple records for some orders due to split payments, multiple installments, or repeated payment entries.
To reduce data granularity and create a more streamlined dataset, payment records were grouped based on:
order_id
payment_type
Within each group, the payment installment values were aggregated by summing the individual installment amounts/counts.
This transformation consolidated multiple payment records into a single summarized record for each order and payment type combination.
Reduced data redundancy and improved the efficiency of downstream analysis.
Simplified payment-related reporting by providing a clearer view of the total payment activity associated with each order.
Facilitated easier analysis of payment methods, installment trends, and overall transaction patterns.
2. Removed Invalid order_id Records Using the Orders Table
Validated the payment records by comparing the order_id values in the payments table against the cleaned and validated orders table.
Performed a join operation between the payments table and orders table using order_id as the common key.
Retained only those payment records whose order_id existed in the valid orders dataset.
Removed payment records associated with:
Invalid orders
Deleted orders
Orders that failed business validation checks during the orders table cleaning process
This ensured referential integrity between the payments and orders datasets.
Eliminated orphan payment records that could lead to inaccurate financial reporting and analysis.
Improved data consistency and reliability across related tables within the database.

In [228]:
payments.sample(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
20150,e9007e380558105d9508dbc5b6e3ec59,1,credit_card,3,151.880005
23484,a03ad7057fae696dd18f8967826c209f,2,voucher,1,44.410000
85127,1f195e1a5568ef0f4153d95a671a0f91,1,boleto,1,80.199997
6212,b2f4a8f4cbaf1ca08d4454b73851e754,1,credit_card,3,103.330002
22527,a4ce473749f3d49434d83a58607881b6,1,debit_card,1,162.250000
90904,c1631d116303f21075fc2d55b4c6e7b0,1,credit_card,1,54.520000
51904,3953c29762c8e3546dfb140558084530,1,boleto,1,173.330002
15762,1f1d5024dbbfd1773a41998b6097e1cb,1,credit_card,3,70.139999
41599,341279a01a590f7c1f08a7ba841d2161,1,credit_card,1,58.220001
7752,84b29272f58b0da99f8884fd87681725,1,credit_card,4,135.220001


In [229]:
payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  str    
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  str    
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 8.1 MB


In [230]:
payments.isna().sum()

order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [231]:
payments[payments['order_id']=='1389d3b1fab26d87e40c382d11a8ac3c']

,order_id,payment_sequential,payment_type,payment_installments,payment_value
87639,1389d3b1fab26d87e40c382d11a8ac3c,1,credit_card,1,15.260000
94171,1389d3b1fab26d87e40c382d11a8ac3c,2,credit_card,4,41.900002


In [232]:
payments['order_id'].duplicated().sum()

np.int64(4446)

In [233]:
payments['payment_type'].unique()

<ArrowStringArray>
['credit_card', 'boleto', 'voucher', 'debit_card', 'not_defined']
Length: 5, dtype: str

In [234]:
payments = payments.groupby(['order_id','payment_type'])['payment_value'].sum().reset_index()

In [235]:
payments.sample(10)

,order_id,payment_type,payment_value
81217,cc852cf36cf64a124456f5f10b2142ae,credit_card,872.169983
20855,34b26ad88bd9eea1ef3628bbbc624e8b,credit_card,89.940002
94712,ee5cc68a9c849b21b595655a944b417c,credit_card,73.169998
74634,bbeeae69169758cd2d24b78df914a56a,voucher,54.500000
87983,dd2be9f213e992cb42d67cb32952f8f0,boleto,95.209999
3093,07bfea8893b7262f7d61cfb2068b12c3,credit_card,48.209999
52308,83fae28795d7d8f02ff4edb1e9e78226,credit_card,50.500000
81927,ce48e13a260d1585d74a5c4e0a830e6c,boleto,45.000000
57028,904a8c5eecb566b06d4737b0cde3e0ab,voucher,65.090000
96250,f245cbeb5e2f1bb0fe643069bb2eee06,credit_card,98.349998


In [236]:
payments.groupby('order_id')['order_id'].count().sort_values(ascending=False)

order_id
42113ccffae542a6aee6b921c765d59e    2
53177d318c723e378c2a2aa1e9b2ea8e    2
54220fcc516cabe9ec84b210c0765ef2    2
1d251ab94983c4adb11e4b168abb1439    2
6a188f1161967f21d6b80a19a283dfec    2
                                   ..
fff1e3e76b816bfe8ef16678cc53c643    1
fff2cdc825f9fc0ba3c04227cfa02303    1
fff2e9e3aa8644e19710216b4ef53ab2    1
fff3983dfa3c5a0d752d8d17baa406a0    1
fff60e5408a9dd1e92ee30023052af30    1
Name: order_id, Length: 99440, dtype: int64

In [237]:
payments.query("order_id == 'd6e320ab3eb91f810c2a3296998bdcc8'")

,order_id,payment_type,payment_value
85459,d6e320ab3eb91f810c2a3296998bdcc8,credit_card,96.610001
85460,d6e320ab3eb91f810c2a3296998bdcc8,voucher,42.040001


In [238]:
payments['order_id'].duplicated().sum()

np.int64(2246)

In [239]:
payments = payments[payments['order_id'].isin(orders['order_id'])]

In [240]:
payments.shape

(99618, 3)

In [241]:
payments['order_id'].nunique()

97420

In [242]:
orders['order_id'].nunique()

97421

## **Olist Items List**

**Order Items (Order List) Table Cleaning – Data Preprocessing Steps**
1. Reduced Row Granularity by Aggregating Order Item Information
The order items table contained multiple records for the same order because a single order could include:
Multiple products
Multiple quantities of the same product
Products supplied by different sellers
To create a more structured and analysis-friendly dataset, the data was grouped using:
order_id
product_id
seller_id
After grouping, the following aggregations were performed:
Summed the product price to calculate the total value of each product within an order.
Summed the freight value (shipping cost) to determine the total shipping cost associated with each product in an order.
Calculated the total quantity of each product purchased within the order.
This process consolidated multiple item-level records into a summarized representation for each unique combination of order, product, and seller.
Reduced unnecessary data duplication and lowered dataset granularity while preserving essential business information.
Improved the efficiency of product-level sales analysis, revenue calculations, shipping cost evaluation, and seller performance reporting.
Enabled a clearer understanding of the total quantity and value of products purchased within each order.
2. Removed Invalid order_id Records Using the Orders Table
Validated all order item records against the cleaned and verified orders table.
Performed a join operation between the order items table and orders table using order_id as the primary matching key.
Retained only those order item records that were associated with valid orders.
Removed records linked to:
Invalid orders
Deleted orders
Orders that failed business validation and data quality checks during the orders table cleaning process
This step ensured referential integrity between the order items table and the orders table.
Eliminated orphan records that could lead to inaccurate sales, revenue, and fulfillment analyses.
Improved consistency across related datasets and ensured that every order item record corresponded to a legitimate order transaction.

In [243]:
olist_items.sample(10)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
6897,0f9cc7808799d149c260814488be8e26,1,b53f20c2b12a4b9821ce57f46a7d1cae,4d6d651bd7684af3fffabd5f08d12e5a,2018-03-15 22:35:26,49.900002,18.230000
62320,8e4f9161e403e220dee5b8eb22f0a3c2,1,5457c026a643626249213e1e39c31d10,a888faf2d1baececa6baf9c3d603ee1f,2017-11-06 18:07:03,259.899994,19.389999
28799,4188cfb6f4039cb4b50ee591cec8086f,1,b0a954d13efe24ba6513fad99bea2967,6b243f80ed07b10f0e8aa0f21a205f3c,2017-11-30 11:55:06,124.900002,19.110001
69728,9f226854839600f3e70c4faeb94e8790,1,666696771a5dd7a28816eab47b70d966,8bb48dc19fccaa8613b6229bf7f452a2,2017-11-28 11:46:54,118.000000,18.080000
40215,5b8b540b5934e4af4790f609af380cf8,4,42e3e1a52d783ace62d11a17c88c1324,897060da8b9a21f655304d50fd935913,2018-03-02 12:48:38,19.690001,16.110001
28195,4034b68eda61d53e7c3b417c10e8b252,1,c06b0271681be1b35ce57c3fc66a9e53,4a3ccda38b2129705f3fb522db62ca31,2018-08-15 09:35:18,155.899994,19.190001
101358,e5e7f892785fbe21a339bcd77efdae86,1,62744bf012b60074e5fa0fcd657cbd34,30a2f535bb48308f991d0b9ad4a8c4bb,2017-12-07 21:24:08,74.900002,8.720000
33754,4c6b8f5bdbbce6acce7f9193da57a34f,1,f1c7f353075ce59d8a6f3cf58f419c9c,37be5a7c751166fbc5f8ccba4119e043,2017-08-24 22:04:43,200.000000,15.010000
52631,77c374d0a559e948d26ddd439b558558,1,f264c1d9b20b5e4a340254d0405e613b,7a67c85e85bb2ce8582c35f2203ad736,2017-12-20 08:52:15,72.989998,13.530000
39620,5a176f6453eb05d7087014b012940727,3,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2017-11-29 19:32:32,49.000000,16.100000


In [244]:
olist_items.shape

(112650, 7)

In [245]:
olist_items = olist_items[olist_items['order_id'].isin(orders['order_id'])]

In [246]:
olist_items.sort_values(by=['order_item_id'],ascending=False)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57317,8272b63d03f5f79c56e9e4120aec44ef,21,79ce45dbc2ea29b22b5a261bbb7b7ee7,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,7.800000,6.570000
57316,8272b63d03f5f79c56e9e4120aec44ef,20,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.200000,7.890000
75122,ab14fdcfbe524636d65ee38360e22ce8,20,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,2017-08-30 14:30:23,98.699997,14.440000
11951,1b15974a0141d54e36626dca3fdc731a,20,ee3d532c8a438679776d222e997606b3,8e6d7754bc7e0f22c96d255ebda59eba,2018-03-01 02:50:48,100.000000,10.120000
75121,ab14fdcfbe524636d65ee38360e22ce8,19,9571759451b1d780ee7c15012ea109d4,ce27a3cc3c8cc1ea79d11e561e9bebb6,2017-08-30 14:30:23,98.699997,14.440000
...,...,...,...,...,...,...,...
112638,fffb0b1a50e65c449020434fa835e078,1,e7be84ea9462aac5e2b0b08eb35ba7f1,36a968b544695394e4e9d7572688598f,2017-04-28 16:45:12,4.900000,10.960000
112637,fffa82886406ccf10c7b4e35c4ff2788,1,bbe7651fef80287a816ead73f065fc4b,8f2ce03f928b567e3d56181ae20ae952,2017-12-22 17:31:42,229.899994,44.020000
112636,fff90cdcb3b2e6cfb397d05d562fd3fe,1,764292b2b0f73f77a0272be03fdd45f3,bd23da7354813347129d751591d1a6e2,2017-11-30 10:11:28,89.900002,11.830000
112649,fffe41c64501cc87c801fd61db3f6244,1,350688d9dc1e75ff97be326363655e01,f7ccf836d21b2fb1de37564105216cc1,2018-06-12 17:10:13,43.000000,12.790000


In [247]:
olist_items.query("order_id == '8272b63d03f5f79c56e9e4120aec44ef'")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [248]:
items = olist_items.groupby(['order_id','product_id','seller_id','shipping_limit_date']).agg(price=('price','sum'),freight_value=('freight_value','sum'),quantity=('product_id','count')).reset_index()

In [249]:
olist_items.query("order_id == '8272b63d03f5f79c56e9e4120aec44ef'")

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
57297,8272b63d03f5f79c56e9e4120aec44ef,1,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57298,8272b63d03f5f79c56e9e4120aec44ef,2,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57299,8272b63d03f5f79c56e9e4120aec44ef,3,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57300,8272b63d03f5f79c56e9e4120aec44ef,4,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57301,8272b63d03f5f79c56e9e4120aec44ef,5,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57302,8272b63d03f5f79c56e9e4120aec44ef,6,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57303,8272b63d03f5f79c56e9e4120aec44ef,7,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57304,8272b63d03f5f79c56e9e4120aec44ef,8,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57305,8272b63d03f5f79c56e9e4120aec44ef,9,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89
57306,8272b63d03f5f79c56e9e4120aec44ef,10,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,1.2,7.89


In [250]:
items.query("order_id == '8272b63d03f5f79c56e9e4120aec44ef'")

,order_id,product_id,seller_id,shipping_limit_date,price,freight_value,quantity
51288,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,12.0,78.899999,10
51289,8272b63d03f5f79c56e9e4120aec44ef,270516a3f41dc035aa87d220228f844c,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,12.0,78.899999,10
51290,8272b63d03f5f79c56e9e4120aec44ef,79ce45dbc2ea29b22b5a261bbb7b7ee7,2709af9587499e95e803a6498a5a56e9,2017-07-21 18:25:23,7.8,6.570000,1


In [251]:
payments.query("order_id == '8272b63d03f5f79c56e9e4120aec44ef'")

,order_id,payment_type,payment_value
51713,8272b63d03f5f79c56e9e4120aec44ef,credit_card,196.110001


In [252]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 100918 entries, 0 to 100917
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             100918 non-null  str           
 1   product_id           100918 non-null  str           
 2   seller_id            100918 non-null  str           
 3   shipping_limit_date  100918 non-null  datetime64[us]
 4   price                100918 non-null  float64       
 5   freight_value        100918 non-null  float64       
 6   quantity             100918 non-null  int64         
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 14.7 MB


In [253]:
items.isna().sum()

order_id               0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
quantity               0
dtype: int64

In [254]:
items.sample(10)

,order_id,product_id,seller_id,shipping_limit_date,price,freight_value,quantity
45341,7312e901d22a5fa199104da9e0996c2d,703d742887bb9267f89b675608ba7aa0,640e21a7d01df7614a3b4923e990d40c,2018-08-20 17:49:21,56.099998,13.970000,1
6092,0f504e88d6d1e382941f38d3724ec4ed,926e1da29dd30de8c9aa8220815f2d85,68f86ba270525243e68ae74044f992b9,2018-05-13 21:35:21,89.000000,15.500000,1
68460,ae2882361453fcb73c3b3fcca0eb9a62,646ce272adff96b1ace27e81c9bdf29d,dbc22125167c298ef99da25668e1011f,2017-11-21 22:11:21,126.900002,14.620000,1
43883,6f6471df0c58534cf3912656d77d46dd,a126cf2ba6e20ffef685352e40945a68,a3e9a2c700480d9bb01fba070ba80a0e,2018-06-19 11:01:06,67.000000,19.660000,1
19136,30e92ef45b77af51660f221fbb2b94e3,cf9eef3269d316b89f8539aafc0db5b4,e0878efa0e0b7e5313ac0b43bc04c081,2018-01-19 15:36:35,159.899994,17.180000,1
7312,127808959bd2d08cdefdf3015fa205a2,efa490145b5cc82438b6a6de691535fa,8d956fec2e4337affcb520f56fd8cbfd,2018-05-03 15:53:37,46.990002,19.040001,1
87409,dd50addabe54a9c924c82e73c4daac41,e24f73b7631ee3fbb2ab700a9acaa258,0cbcee27c791afa0cdcb08587a2013a8,2018-05-04 02:30:46,115.000000,23.389999,1
66080,a853cdd8003fb8372f09eefdb15720f8,31c79131e883e5fd8c4c85fe9f7d2bb2,ea8482cd71df3c1969d7b9473ff13abc,2018-05-30 17:30:42,49.980000,31.900000,2
25148,4007c8123697940f0574798a18c6ad29,dad34da019832ea09413a1800947bec3,39d54ff918774174706fb065d7f9dc07,2018-06-04 19:10:19,31.500000,19.320000,1
56372,8fc40ab512fd704815938e379ac9c36f,e60e632d39df1e129df5b7f0245716f7,855668e0971d4dfd7bef1b6a4133b41b,2017-09-14 14:30:26,75.000000,17.780001,1


In [255]:
items.shape

(100918, 7)

## **Reviews**

**Reviews Table Cleaning – Data Preprocessing Steps**
1. Reduced Row Granularity by Aggregating Review Information
The reviews table contained multiple review records associated with certain orders, resulting in a higher level of data granularity.
To create a more concise and analysis-friendly dataset, review records were grouped based on order_id.
For each order, the review scores were aggregated by calculating the average review rating.
This transformation produced a single representative review score for each order, eliminating the need to analyze multiple review entries separately.
Reduced data redundancy while preserving the overall customer satisfaction information associated with each order.
Simplified customer feedback analysis and enabled easier integration with other order-related datasets.
Improved the efficiency of reporting, visualization, and performance measurement activities.
2. Removed Invalid order_id Records Using the Orders Table
Validated all review records by comparing their order_id values against the cleaned and verified orders table.
Performed a join operation between the reviews table and orders table using order_id as the common key.
Retained only those review records that corresponded to valid orders.
Removed review entries associated with:
Invalid orders
Deleted orders
Orders that failed validation checks during the orders table cleaning process
This step ensured referential integrity between the reviews and orders datasets.
Eliminated orphan review records that could distort customer satisfaction metrics and business insights.
Improved consistency and reliability across related tables within the data model.
Ensured that every review included in the analysis was linked to a legitimate customer transaction.

In [256]:
query ="select * from olist_order_reviews_dataset"
reviews= pd.read_sql(query, conn)

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\2438451688.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews= pd.read_sql(query, conn)


In [257]:
reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01,2018-07-02 12:59:13


In [258]:
reviews['order_id'].duplicated().sum()

np.int64(551)

In [259]:
reviews['order_id'].value_counts().sort_values(ascending=False).head(5)

order_id
c88b1d1b157a9999ce368f218a407141    3
df56136b8031ecd28e200bb18e6ddb2e    3
03c939fd7fd3b38f8485a0f95798f1f6    3
8e17072ec97ce29f0e1f111e598b0c85    3
cf73e2cb1f4a9480ed70c154da3d954a    2
Name: count, dtype: int64

In [260]:
reviews.query("order_id == 'c88b1d1b157a9999ce368f218a407141'")

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
1985,ffb8cff872a625632ac983eb1f88843c,c88b1d1b157a9999ce368f218a407141,3,NaN,NaN,2017-07-22,2017-07-26 13:41:07
82525,202b5f44d09cd3cfc0d6bd12f01b044c,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-22,2017-07-26 13:40:22
89360,fb96ea2ef8cce1c888f4d45c8e22b793,c88b1d1b157a9999ce368f218a407141,5,NaN,NaN,2017-07-21,2017-07-26 13:45:15


In [261]:
items.query("order_id == 'c88b1d1b157a9999ce368f218a407141'")

,order_id,product_id,seller_id,shipping_limit_date,price,freight_value,quantity
79086,c88b1d1b157a9999ce368f218a407141,b1acb7e8152c90c9619897753a75c973,cc419e0650a3c5ba77189a1882b7556a,2017-07-26 22:50:12,34.990002,7.78,1


In [262]:
reviews = reviews.groupby(['order_id'])['review_score'].mean().reset_index()

In [263]:
reviews

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0
...,...,...
98668,fffc94f6ce00a00581880bf54a75a037,5.0
98669,fffcd46ef2263f404302a634eb57f7eb,5.0
98670,fffce4705a9662cd70adb13d4a31832d,5.0
98671,fffe18544ffabc95dfada21779c9644f,5.0


In [264]:
reviews = reviews[reviews['order_id'].isin(orders['order_id'])]

In [265]:
reviews.shape

(96676, 2)

## **Final Python Data Cleaning Script**

## **Customer Table**

In [266]:
import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

#Cleaning Functions
def converting_to_english(temp_df,column):
    "Converts to english language"
    temp_df[column] = temp_df[column].apply(unidecode)

    return temp_df


def keep_text(temp_df,column_list):
    "Removes unwanted spaces and special characters from textual columns"

    for col in column_list:
        temp_df[col] = temp_df[col].str.strip()
        temp_df[col] = temp_df[col].str.replace(r'[\s+]',' ',regex=True)
        temp_df[col] = temp_df[col].str.replace(r'[^A-Za-z\s]','',regex=True)

    return temp_df 
   
#Importing customer data
query="select * from olist_customers_dataset"
customers= pd.read_sql(query, conn)

#Converting to title case
customers['customer_city'] = customers['customer_city'].str.title()

#Converting Portugese to English
customers = converting_to_english(customers,'customer_city')

#Removing special characters
customers = keep_text(customers,['customer_city'])

#Droping column
customers.drop(columns=['customer_unique_id'],inplace=True)

#Remving spaces
customers['customer_id'] = customers['customer_id'].str.strip()

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\268604312.py:38: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  customers= pd.read_sql(query, conn)


## **Geolocation Table**

In [267]:
import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

#Cleaning Function
def converting_to_english(temp_df,column):
    "Converts to english language"
    temp_df[column] = temp_df[column].apply(unidecode)

    return temp_df

def keep_text(temp_df,column_list):
    "Removes unwanted spaces and special characters from textual columns"

    for col in column_list:
        temp_df[col] = temp_df[col].str.strip()
        temp_df[col] = temp_df[col].str.replace(r'[\s+]',' ',regex=True)
        temp_df[col] = temp_df[col].str.replace(r'[^A-Za-z\s]','',regex=True)

    return temp_df



#Importing Geolocation Data
query="select * from olist_geolocation_dataset"
geolocation= pd.read_sql(query, conn)


#Droping columns
geolocation.drop(columns=['geolocation_lat','geolocation_lng'],inplace=True)

#Converting to title case
geolocation['geolocation_city'] = geolocation['geolocation_city'].str.title()

#Converting from Portugese to English
geolocation = converting_to_english(geolocation,'geolocation_city')

#Removing special characters
geolocation = keep_text(geolocation,['geolocation_city'])

#Special case removel
geolocation["geolocation_city"]= geolocation["geolocation_city"].str.replace("Sao Joao Do Pau D Alho","Sao Joao Do Pau DAlho")

#Droping Duplicates
geolocation.drop_duplicates(inplace=True)

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\3750589807.py:39: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  geolocation= pd.read_sql(query, conn)


## **Products Table**

In [268]:
import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

#Importing Data
query ="select * from olist_products_dataset"
products= pd.read_sql(query, conn)

query ="select * from product_category_name_translation"
product_category_name_translation= pd.read_sql(query, conn)

#Data Cleaning
product_category_name_translation.drop(index=0,inplace=True)

products.drop(columns=['product_name_lenght','product_description_lenght', 'product_weight_g',
       'product_length_cm', 'product_height_cm','product_width_cm'],inplace=True)

products = products.merge(product_category_name_translation,on='product_category_name',how='left')

#Replacing 'pc_gamer' with 'Gaming PC'
products.loc[products['product_category_name']=='pc_gamer','product_category_name_english']='Gaming PC'


#Replacing 'portateis_cozinha_e_preparadores_de_alimentos' with 'Portable Kitchen Appliances'
products.loc[products['product_category_name']=='portateis_cozinha_e_preparadores_de_alimentos','product_category_name_english']='Portable Kitchen Appliances'

products = products.drop(columns='product_category_name').rename(columns={'product_category_name_english':'product_category_name'}).reindex(columns=['product_id','product_category_name','product_photos_qty'])

products['product_category_name'] = products['product_category_name'].str.title().str.replace('_',' ').fillna('Other')

products['product_photos_qty'] = products['product_photos_qty'].fillna(1)

products['product_id'] = products['product_id'].str.strip()

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\677641321.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  products= pd.read_sql(query, conn)
C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\677641321.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  product_category_name_translation= pd.read_sql(query, conn)


## **Sellers Table**

In [269]:
import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

#Importing data
query ="select * from olist_sellers_dataset"
sellers= pd.read_sql(query, conn)

#Data Cleaning
sellers['seller_id'] = sellers['seller_id'].str.strip()

sellers['seller_city'] = (
sellers['seller_city'].str.replace(r'[/,\\-]\s*\w*','',regex=True)
.str.strip()
.str.replace('04482255','Not Known')
.str.replace('vendas@creditparts.com.br','Not known')
.str.title()    
.str.replace(r'Sao Paulo  Paulo|Sao Pauo|Sao Paulo Sp|Sao Paulop|Sao Paluo','Sao Paulo',regex=True)
)

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\3810604289.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sellers= pd.read_sql(query, conn)


## **Orders Table and Payments Table**

In [270]:
import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

#Cleaning Function
def remove_invalid_order_id_rows(row):
    delivery_date = row['order_delivered_customer_date']
    check1 = row['order_purchase_timestamp']<=row['order_approved_at']
    check2 = row['order_approved_at']<=row['order_delivered_carrier_date']
    check3 = row['order_delivered_carrier_date']<=row['order_delivered_customer_date']

    if row['order_status']=='delivered':
        if check1 and check2 and check3:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='shipped':
        if check1 and check2:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='processing':
        if check1:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='canceled':
        if pd.isna(delivery_date):
            return 'Yes'
        else:
            return 'No'
    else:
        return 'Yes' 

#Importing Data
query ="select * from olist_orders_dataset"
orders= pd.read_sql(query, conn)

query ="select * from olist_order_payments_dataset"
payments= pd.read_sql(query, conn)

#Orders Table Cleaning
orders = orders[orders['order_status']!='unavailable']

orders['order_status'] = orders['order_status'].str.replace(r'invoiced|approved','processing',regex=True)

orders['Valid'] = orders.apply(remove_invalid_order_id_rows,axis=1)

orders = orders.loc[orders['Valid'] == 'Yes']

orders.drop(columns = ['Valid','order_estimated_delivery_date'],inplace=True)

orders['order_id'] = orders['order_id'].str.strip()


#Payments Table Cleaning
payments = payments.groupby(['order_id','payment_type'])['payment_value'].sum().reset_index()

payments = payments[payments['order_id'].isin(orders['order_id'])]

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\4070182969.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders= pd.read_sql(query, conn)
C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\4070182969.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  payments= pd.read_sql(query, conn)


## **Order Items Table**

In [271]:
import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

#Cleaning Function
def remove_invalid_order_id_rows(row):
    delivery_date = row['order_delivered_customer_date']
    check1 = row['order_purchase_timestamp']<=row['order_approved_at']
    check2 = row['order_approved_at']<=row['order_delivered_carrier_date']
    check3 = row['order_delivered_carrier_date']<=row['order_delivered_customer_date']

    if row['order_status']=='delivered':
        if check1 and check2 and check3:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='shipped':
        if check1 and check2:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='processing':
        if check1:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='canceled':
        if pd.isna(delivery_date):
            return 'Yes'
        else:
            return 'No'
    else:
        return 'Yes'


#Importing Data
query ="select * from olist_orders_dataset"
orders= pd.read_sql(query, conn)

query ="select * from olist_order_items_dataset"
olist_items= pd.read_sql(query, conn)

#Orders Table Cleaning
orders = orders[orders['order_status']!='unavailable']

orders['order_status'] = orders['order_status'].str.replace(r'invoiced|approved','processing',regex=True)

orders['Valid'] = orders.apply(remove_invalid_order_id_rows,axis=1)

orders = orders.loc[orders['Valid'] == 'Yes']

orders.drop(columns = ['Valid','order_estimated_delivery_date'],inplace=True)

orders['order_id'] = orders['order_id'].str.strip()


#Items Table cleaning
items = olist_items.groupby(['order_id','product_id','seller_id','shipping_limit_date']).agg(price=('price','sum'),freight_value=('freight_value','sum'),quantity=('product_id','count')).reset_index()

items = items[items['order_id'].isin(orders['order_id'])]

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\4047210375.py:51: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders= pd.read_sql(query, conn)
C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\4047210375.py:54: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  olist_items= pd.read_sql(query, conn)


## **Reviews Table**

In [272]:
import numpy as np
import pandas as pd
import pyodbc
from unidecode import unidecode

server="SUMITBERDE\SQLEXPRESS"
database="Ecommerce Dashboard"

connection_string=(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
)

conn = pyodbc.connect(connection_string)

#Cleaning Function
def remove_invalid_order_id_rows(row):
    delivery_date = row['order_delivered_customer_date']
    check1 = row['order_purchase_timestamp']<=row['order_approved_at']
    check2 = row['order_approved_at']<=row['order_delivered_carrier_date']
    check3 = row['order_delivered_carrier_date']<=row['order_delivered_customer_date']

    if row['order_status']=='delivered':
        if check1 and check2 and check3:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='shipped':
        if check1 and check2:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='processing':
        if check1:
            return 'Yes'
        else:
            return 'No'
    elif row['order_status']=='canceled':
        if pd.isna(delivery_date):
            return 'Yes'
        else:
            return 'No'
    else:
        return 'Yes' 

#Importing Data
query ="select * from olist_orders_dataset"
orders= pd.read_sql(query, conn)

query ="select * from olist_order_reviews_dataset"
reviews= pd.read_sql(query, conn)

#Orders Table Cleaning
orders = orders[orders['order_status']!='unavailable']

orders['order_status'] = orders['order_status'].str.replace(r'invoiced|approved','processing',regex=True)

orders['Valid'] = orders.apply(remove_invalid_order_id_rows,axis=1)

orders = orders.loc[orders['Valid'] == 'Yes']

orders.drop(columns = ['Valid','order_estimated_delivery_date'],inplace=True)

orders['order_id'] = orders['order_id'].str.strip()


#Reviews Table Cleaning
reviews = reviews.groupby(['order_id'])['review_score'].mean().reset_index()
        
reviews = reviews[reviews['order_id'].isin(orders['order_id'])]

C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\2452898083.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  orders= pd.read_sql(query, conn)
C:\Users\sumit\AppData\Local\Temp\ipykernel_23432\2452898083.py:53: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  reviews= pd.read_sql(query, conn)
